# Aula 7 — Bioinformática Estrutural: prática em Python (Google Colab)

**Disciplina:** Introdução à Bioinformática — PPGED / IME

Este caderno é o **par prático** do roteiro da Aula 7. Ele percorre, com **dados públicos** e **software livre**, o caminho completo da bioinformática estrutural: do arquivo do PDB ao docking validado, passando por um modelo previsto pelo AlphaFold e terminando na leitura de uma trajetória de dinâmica molecular.

**Nosso alvo:** a **acetilcolinesterase humana (AChE)** — a enzima inibida pelos agentes neurotóxicos organofosforados e alvo de antídotos e de fármacos para Alzheimer. Estrutura **4EY7**: raios X, 2,35 Å, R-free 0,211, em complexo com a **donepezila**.

| Bloco | O que você vai fazer | Conceito da teoria |
|---|---|---|
| **A** | Explorar a 4EY7 no navegador (sem código) | PDB, resolução, R-free, cadeias, representações |
| **B** | Comparar o modelo do **AlphaFold** com o cristal | pLDDT, PAE, enovelamento global × qualidade local |
| **C** | Preparar receptor e ligante, **dockar** com o AutoDock Vina e **validar por redocking** | Pose, score, caixa de busca, RMSD, protonação, cargas |
| **D** | Ler uma trajetória de **dinâmica molecular** | RMSD, raio de giro, RMSF, convergência |

---

### Primeira vez no Colab? Leia isto (2 minutos)

- Um **caderno** (*notebook*) mistura **texto** (como este) e **células de código** (caixas cinzas).
- Para **executar** uma célula: clique nela e pressione **Shift + Enter** (ou clique no ▶ à esquerda).
- Execute as células **em ordem, de cima para baixo**. Cada uma usa o que a anterior preparou.
- Linhas que começam com `#` são **comentários**: explicações para humanos, que o computador ignora. **Leia os comentários** — eles são o material didático.
- Linhas que começam com `!` são **comandos de terminal**, executados na máquina do Google.
- Você **não precisa** entender cada linha. Foque no *que* cada bloco faz e no *resultado* que ele mostra.
- Se algo travar: **Ambiente de execução → Reiniciar e executar tudo**.

> **Tempo estimado:** Bloco A ≈ 12 min (navegador) · B ≈ 15 min · C ≈ 33 min · D ≈ 15 min.

### A regra de ouro deste caderno

Depois de quase todo passo há uma **conferência numérica barata** — um número que você compara com o valor esperado. Isso não é zelo excessivo: em bioinformática estrutural, **os erros que mais custam não quebram o programa**, apenas produzem um resultado plausível e errado. Se um número não bater, **pare e corrija antes de seguir**.


---
## 0. Preparação do ambiente

Python sozinho sabe fazer contas e manipular texto. Para tarefas especializadas usamos **bibliotecas** e **programas externos**. Hoje:

| Ferramenta | O que é | Para que serve aqui |
|---|---|---|
| **AutoDock Vina** | Programa de **docking molecular**, código aberto (Scripps). O mais usado do mundo | Encaixar o ligante no sítio ativo e pontuar a pose |
| **Open Babel** | "Canivete suíço" da química computacional, código aberto | Converter formatos, adicionar hidrogênios, calcular cargas, medir RMSD entre moléculas |
| **RDKit** | Biblioteca de **quimioinformática**, código aberto | Transformar o texto SMILES em uma molécula 3D |
| **Biopython** | Biblioteca de bioinformática | Ler arquivos PDB, alinhar sequências, sobrepor estruturas |
| **MDAnalysis** | Biblioteca de análise de **trajetórias** de dinâmica molecular | RMSD, raio de giro e RMSF do Bloco D |
| **NumPy · pandas · Matplotlib** | Contas, tabelas e gráficos | O de sempre |

A célula abaixo instala tudo. **Leva de 1 a 2 minutos** — comece a rodar e leia o Bloco A enquanto espera.


In [ ]:
# ===================================================================
# INSTALAÇÃO (1 a 2 minutos). O "-q" deixa a saída silenciosa.
# ===================================================================

# AutoDock Vina: baixamos o binário oficial publicado no GitHub do projeto.
VINA_VERSAO = "1.2.5"
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v{VINA_VERSAO}/vina_{VINA_VERSAO}_linux_x86_64 -O vina
!chmod +x vina

# Bibliotecas Python
!pip install -q rdkit biopython MDAnalysis MDAnalysisTests

# Open Babel: tentamos o pacote do sistema (mais rápido no Colab)
!apt-get -qq install -y openbabel > /dev/null 2>&1

print("Instalação terminada. Rode a próxima célula para conferir.")

In [ ]:
# ===================================================================
# CONFERÊNCIA DO AMBIENTE + ajuste do Open Babel, se necessário
# ===================================================================
import os, re, glob, shutil, subprocess, warnings
warnings.filterwarnings("ignore")          # silencia avisos técnicos que não nos interessam


def garantir_openbabel():
    """Garante que os comandos 'obabel' e 'obrms' existam.

    Plano A: o pacote do sistema (instalado na célula anterior).
    Plano B: a versão distribuída pelo pip ('openbabel-wheel'), caso o apt falhe.
             Nesse caso precisamos indicar à mão onde estão os plugins da biblioteca,
             através das variáveis de ambiente BABEL_LIBDIR e BABEL_DATADIR.
    """
    if shutil.which("obabel") and shutil.which("obrms"):
        return "pacote do sistema (apt)"

    subprocess.run("pip install -q openbabel-wheel", shell=True)
    import openbabel as _ob
    base = os.path.dirname(_ob.__file__)
    os.environ["PATH"] = os.path.join(base, "bin") + os.pathsep + os.environ["PATH"]
    for variavel, subpasta in [("BABEL_LIBDIR", "lib/openbabel"), ("BABEL_DATADIR", "share/openbabel")]:
        achados = glob.glob(os.path.join(base, subpasta, "*"))
        if achados:
            os.environ[variavel] = achados[0]
    return "pacote do pip (openbabel-wheel)"


origem_obabel = garantir_openbabel()

print("--- versões (anote-as: elas vão para a seção de métodos do seu artigo) ---")
!./vina --version | head -2
!obabel -V
print("Open Babel veio do:", origem_obabel)
print("obrms encontrado em:", shutil.which("obrms"))

In [ ]:
# ===================================================================
# IMPORTAÇÕES, APARÊNCIA DOS GRÁFICOS E UMA FUNÇÃO PARA BAIXAR ARQUIVOS
# ===================================================================
import time, warnings
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")   # silencia avisos técnicos que não nos interessam hoje

# Cores dos slides da aula (laranja e azul-petróleo continuam distinguíveis
# para pessoas com daltonismo, porque diferem também em luminosidade)
COR_A, COR_B, COR_NS = "#B35A2A", "#2F6B7A", "#B9B2A4"
TINTA, TINTA_SUAVE = "#12232E", "#43525C"

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": TINTA_SUAVE, "axes.labelcolor": TINTA,
    "xtick.color": TINTA_SUAVE, "ytick.color": TINTA_SUAVE,
    "axes.grid": True, "grid.color": "#E6E1D6", "grid.linewidth": 0.8,
})
pd.set_option("display.width", 160)


def baixar(url, nome_arquivo, tentativas=3):
    """Baixa um arquivo e salva com o nome indicado. Devolve True se deu certo.

    Se o arquivo já existir (porque você rodou a célula antes), não baixa de novo.
    Tenta algumas vezes: servidores públicos às vezes falham por instantes.
    """
    if os.path.exists(nome_arquivo) and os.path.getsize(nome_arquivo) > 0:
        print(f"✔ {nome_arquivo} já está aqui (não precisei baixar de novo)")
        return True
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, timeout=180)
            resposta.raise_for_status()
            with open(nome_arquivo, "wb") as arquivo:
                arquivo.write(resposta.content)
            print(f"✔ baixado: {nome_arquivo} ({len(resposta.content)/1e6:.2f} MB) de {url}")
            return True
        except Exception as erro:
            print(f"  tentativa {tentativa} falhou: {erro}")
            time.sleep(2 * tentativa)
    return False


def baixar_estrutura_pdb(codigo):
    """Baixa uma estrutura do PDB, com servidor alternativo.

    Algumas redes institucionais e militares bloqueiam 'files.rcsb.org'.
    Por isso tentamos também o PDBe (o espelho europeu, no EMBL-EBI) e,
    se nada funcionar, explicamos como enviar o arquivo à mão.
    """
    codigo = codigo.upper()
    destino = f"{codigo}.pdb"
    fontes = [
        f"https://files.rcsb.org/download/{codigo}.pdb",                              # RCSB (EUA)
        f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{codigo.lower()}.ent",   # PDBe (Europa)
    ]
    for url in fontes:
        if baixar(url, destino):
            return destino
    print(f"""
    ✖ Não consegui baixar a estrutura {codigo} automaticamente.
      Provavelmente a rede está bloqueando o download. Faça assim:
        1. abra https://www.rcsb.org/structure/{codigo}
        2. clique em 'Download Files' → 'PDB Format'
        3. no Colab, clique no ícone de PASTA (barra lateral esquerda)
           e ARRASTE o arquivo baixado para lá
        4. renomeie-o para '{destino}' se necessário e rode esta célula de novo
    """)
    return None


print("Tudo pronto. Pode seguir.")

---
# BLOCO A — Explorando uma estrutura experimental (12 min)

**Só navegador. Sem código.** Antes de programar qualquer coisa, precisamos *olhar* a estrutura. Visualizar não é enfeite: é a etapa de **controle de qualidade**. Erros grosseiros — cadeia faltando, ligante no lugar errado, duas cópias da proteína — aparecem em 10 segundos na tela e passariam despercebidos no arquivo de texto.

### Passo a passo

1. Acesse <https://www.rcsb.org> e busque por **4EY7**.
2. Na página de resumo, **anote**:

| O que | Valor que você encontrou | Valor esperado |
|---|---|---|
| Método experimental | | raios X |
| Resolução | | 2,35 Å |
| R-free | | 0,211 |
| Organismo | | *Homo sapiens* |
| **Quantas cadeias de polímero (*Macromolecules*)** | | **duas de AChE (A e B) + duas de glicano (C e D)** |
| Ligantes (*Ligands*) | | **E20** (donepezila), NAG, EDO, NO3 |

3. Clique em **3D View** para abrir o **Mol\*** (o visualizador oficial do PDB, que roda dentro do navegador).
4. Faça nesta ordem:
   - Gire a molécula (botão esquerdo) e dê zoom (*scroll*).
   - **Conte as cópias da proteína na tela** — colore por cadeia (*Polymer Chain*). Você deve ver **duas** unidades da AChE, cada uma com sua própria donepezila.
   - Mude a representação para **Cartoon** e identifique hélices α e folhas β.
   - Ative **Surface** e use o modo de **corte** (*clip*) para fatiar a proteína e enxergar a **garganta** por dentro.
   - Na lista de componentes, selecione **uma** molécula de **E20** e clique em *Focus*.
   - Peça para mostrar as **interações** do ligante com os resíduos vizinhos.

### Perguntas de checagem

- **A.1** A donepezila está na superfície ou enterrada?
- **A.2** Quantos resíduos aromáticos (TRP, TYR, PHE) você vê ao redor dela?
- **A.3** Localize **TRP86** (fundo da garganta) e **TRP286** (entrada, sítio periférico). Por que uma molécula que toca os dois extremos seria um inibidor potente?
- **A.4** Olhando as duas cópias na tela: se você calculasse o **ponto médio** entre as duas donepezilas, esse ponto cairia dentro de um sítio ativo? **Guarde a resposta — ela explica o passo C4.**


---
# BLOCO B — E quando não há estrutura experimental? (15 min)

Existem centenas de milhões de sequências de proteínas conhecidas e cerca de 200 mil estruturas experimentais. **Para a maioria das proteínas não há estrutura resolvida.** As duas saídas são a **modelagem por homologia** (SWISS-MODEL) e a **predição por aprendizado profundo** (AlphaFold).

## B1. AlphaFold DB (no navegador, 5 min)

O **AlphaFold DB** (<https://alphafold.ebi.ac.uk>, mantido pelo EMBL-EBI) guarda mais de 200 milhões de estruturas **previstas** — incluindo todo o proteoma humano. Para a maioria das proteínas, o modelo já está pronto e é só baixar.

1. Busque **P22303** (o código da AChE humana no UniProt).
2. Observe o modelo colorido por **pLDDT** — a confiança **local**, de 0 a 100: azul-escuro é alta, laranja é baixa.
3. Note que o **núcleo globular é azul** e os **terminais e alças são laranja**. Por quê? Porque as alças e as pontas realmente não têm uma forma fixa única — pLDDT baixo frequentemente indica **região desordenada**, e não erro do método.
4. Abra o gráfico **PAE** (erro esperado na posição **relativa** entre pares de resíduos) e identifique os blocos — cada bloco é um domínio bem definido.
5. Note também o que o modelo **não** tem: é um **monômero**, sem glicanos, sem ligante, sem águas e sem íons.

## B2. Comparando previsão com realidade (no código, 10 min)

Vamos sobrepor o modelo do AlphaFold à cadeia A do cristal 4EY7 e medir o **RMSD** (*Root Mean Square Deviation*) do esqueleto — a raiz da média dos quadrados das distâncias entre átomos correspondentes, em ångström.

### Uma armadilha de numeração que precisa ser resolvida primeiro

A tentação é parear os resíduos **pelo número**: resíduo 100 do cristal com resíduo 100 do modelo. **Isso dá errado aqui.** A AChE humana é sintetizada com um **peptídeo-sinal** de 31 aminoácidos, que é cortado antes de a proteína ficar pronta:

- o **AlphaFold** numera pela sequência do UniProt, que **inclui** o peptídeo-sinal;
- o **cristal** numera pela proteína **madura**, que não o tem.

Resultado: as duas numerações estão **deslocadas** uma da outra. Parear por número compararia resíduos diferentes e produziria um RMSD enorme e sem sentido — mais um erro que não gera mensagem de erro.

**A solução robusta:** alinhar as duas **sequências** (como na Aula 2) e parear os resíduos **pelo alinhamento**. O código abaixo faz isso e, de brinde, **descobre e imprime o deslocamento** — você vai ver o número aparecer.


In [ ]:
# Baixa o cristal e o modelo previsto
arquivo_cristal = baixar_estrutura_pdb("4EY7")

def baixar_modelo_alphafold(uniprot="P22303"):
    """Baixa o modelo do AlphaFold DB. Pergunta à API qual é o arquivo atual
    (a versão do modelo muda com o tempo) e, se a API falhar, tenta os nomes diretos."""
    destino = f"AF-{uniprot}.pdb"
    try:
        ficha = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot}", timeout=60).json()
        url = ficha[0]["pdbUrl"]
        if baixar(url, destino):
            return destino
    except Exception as erro:
        print("  (API do AlphaFold indisponível:", erro, ")")
    for versao in ("v4", "v3", "v2"):
        if baixar(f"https://alphafold.ebi.ac.uk/files/AF-{uniprot}-F1-model_{versao}.pdb", destino):
            return destino
    print(f"""
    ✖ Não consegui baixar o modelo. Baixe à mão em
      https://alphafold.ebi.ac.uk/entry/{uniprot}  →  'PDB file'
      e arraste o arquivo para o painel de pastas do Colab, com o nome '{destino}'.
    """)
    return None

arquivo_modelo = baixar_modelo_alphafold("P22303")

In [ ]:
# ===================================================================
# SOBREPOSIÇÃO POR ALINHAMENTO DE SEQUÊNCIA (não por número de resíduo!)
# ===================================================================
from Bio.PDB import PDBParser, Superimposer
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner

leitor = PDBParser(QUIET=True)


def carbonos_alfa(caminho_pdb, cadeia="A"):
    """Devolve, para uma cadeia, a lista de (número do resíduo, letra do aminoácido, átomo CA).

    O 'CA' é o carbono alfa: um átomo por aminoácido, que representa o esqueleto da proteína.
    Usar só os CA é o padrão para comparar enovelamentos.
    """
    estrutura = leitor.get_structure("x", caminho_pdb)
    residuos = []
    for residuo in estrutura[0][cadeia]:
        # is_aa descarta água, íons e ligantes; exigimos que o CA exista
        if is_aa(residuo, standard=False) and "CA" in residuo:
            residuos.append((residuo.id[1], seq1(residuo.get_resname()), residuo["CA"]))
    return residuos


def sobrepor(lista_1, lista_2):
    """Alinha as SEQUÊNCIAS das duas listas de resíduos e sobrepõe os CA pareados.

    Devolve (rmsd_global, pares, deslocamento_de_numeracao, desvios_por_residuo).
    """
    seq_1 = "".join(letra for _, letra, _ in lista_1)
    seq_2 = "".join(letra for _, letra, _ in lista_2)

    # Alinhamento global simples: prêmio por igualdade, castigo por diferença e por lacuna
    alinhador = PairwiseAligner(mode="global", match_score=2, mismatch_score=-1,
                                open_gap_score=-11, extend_gap_score=-1)
    alinhamento = alinhador.align(seq_1, seq_2)[0]

    # 'aligned' devolve os trechos que se correspondem; expandimos para pares resíduo a resíduo
    pares = []
    for (i0, i1), (j0, j1) in zip(alinhamento.aligned[0], alinhamento.aligned[1]):
        for k in range(i1 - i0):
            pares.append((lista_1[i0 + k], lista_2[j0 + k]))

    # Superimposer calcula a rotação+translação que minimiza o RMSD e aplica-a
    sup = Superimposer()
    sup.set_atoms([a[2] for a, _ in pares], [b[2] for _, b in pares])
    sup.apply([b[2] for _, b in pares])          # move o segundo conjunto para cima do primeiro

    # Desvio que sobrou em cada resíduo, depois da melhor sobreposição possível
    desvios = pd.DataFrame({
        "residuo_cristal": [a[0] for a, _ in pares],
        "aminoacido":      [a[1] for a, _ in pares],
        "desvio_A":        [float(np.linalg.norm(a[2].coord - b[2].coord)) for a, b in pares],
        # Nos modelos do AlphaFold, o campo do B-factor guarda o pLDDT (0 a 100)
        "pLDDT":           [float(b[2].get_bfactor()) for _, b in pares],
    })

    deslocamentos = [b[0] - a[0] for a, b in pares]
    deslocamento = max(set(deslocamentos), key=deslocamentos.count)   # o valor mais frequente
    return sup.rms, pares, deslocamento, desvios


cristal = carbonos_alfa(arquivo_cristal, "A")
modelo  = carbonos_alfa(arquivo_modelo,  "A")
print(f"Cα na cadeia A do cristal 4EY7 : {len(cristal)}")
print(f"Cα no modelo do AlphaFold      : {len(modelo)}")

rmsd, pares, deslocamento, desvios = sobrepor(cristal, modelo)

print(f"\nResíduos pareados pelo alinhamento: {len(pares)}")
print(f"Deslocamento de numeração detectado: {deslocamento:+d}")
print(f"   (ou seja: o resíduo 1 do cristal é o resíduo {1 + deslocamento} do UniProt —")
print(f"    o peptídeo-sinal cortado tem {deslocamento} aminoácidos)")
print(f"\n>>> RMSD global do esqueleto: {rmsd:.2f} Å")

**Pergunta B.1** — O deslocamento impresso foi 31? Confira na ficha do UniProt **P22303**, seção *PTM/Processing*: qual é o comprimento anotado do peptídeo-sinal?

**Pergunta B.2** — O RMSD global ficou na faixa de 1 a 2 Å? Se sim, o AlphaFold **acertou o enovelamento global** desta proteína.

### Onde o modelo erra — e por que isso importa para o docking

Um RMSD **global** de 1 Å não significa que o modelo serve para tudo. Vamos olhar o desvio **resíduo a resíduo** e cruzá-lo com o **pLDDT**. A expectativa: os resíduos de pLDDT baixo são justamente os de maior desvio.


In [ ]:
fig, eixos = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Painel de cima: o desvio que sobrou em cada resíduo após a sobreposição
eixos[0].plot(desvios["residuo_cristal"], desvios["desvio_A"], lw=1.1, color=COR_A)
eixos[0].axhline(2.0, ls="--", lw=1, color=TINTA_SUAVE)
eixos[0].set_ylabel("desvio (Å)")
eixos[0].set_title("Onde o modelo do AlphaFold difere do cristal (linha tracejada = 2 Å)",
                   loc="left", color=TINTA)

# Painel de baixo: a confiança que o próprio AlphaFold declarou para cada resíduo
eixos[1].plot(desvios["residuo_cristal"], desvios["pLDDT"], lw=1.1, color=COR_B)
eixos[1].axhline(90, ls="--", lw=1, color=TINTA_SUAVE)
eixos[1].axhline(70, ls=":",  lw=1, color=TINTA_SUAVE)
eixos[1].set_ylabel("pLDDT")
eixos[1].set_xlabel("número do resíduo (numeração do cristal)")
eixos[1].set_title("Confiança declarada pelo AlphaFold (tracejado 90 = muito alta; pontilhado 70 = boa)",
                   loc="left", color=TINTA)
plt.tight_layout(); plt.show()

# A correlação entre confiança declarada e erro real
correlacao = desvios["pLDDT"].corr(desvios["desvio_A"])
print(f"Correlação entre pLDDT e desvio real: {correlacao:+.2f}  (esperada NEGATIVA: mais confiança, menos erro)")

# Agora o que de fato importa para o docking: como está o SÍTIO ATIVO?
SITIO_ATIVO = {86: "TRP86 (fundo da garganta)", 202: "TYR202", 203: "SER203 (serina catalítica)",
               286: "TRP286 (sítio periférico)", 337: "TYR337", 447: "HIS447 (tríade catalítica)"}
print("\n--- Qualidade do modelo NOS RESÍDUOS DO SÍTIO ATIVO ---")
tabela_sitio = desvios[desvios["residuo_cristal"].isin(SITIO_ATIVO)].copy()
tabela_sitio["resíduo"] = tabela_sitio["residuo_cristal"].map(SITIO_ATIVO)
print(tabela_sitio[["resíduo", "aminoacido", "desvio_A", "pLDDT"]].to_string(index=False))

print(f"\nDesvio médio no sítio ativo : {tabela_sitio['desvio_A'].mean():.2f} Å")
print(f"Desvio médio na proteína    : {desvios['desvio_A'].mean():.2f} Å")

**Pergunta B.3** — A correlação entre pLDDT e desvio real ficou negativa? Isso significa que o AlphaFold **sabe onde não sabe** — e é por isso que o pLDDT é uma métrica útil, e não um enfeite.

**Pergunta B.4 (a discussão que vale o bloco)** — Se o enovelamento global está certo, **por que ainda é arriscado fazer docking diretamente no modelo do AlphaFold?**

*Resposta a construir:* o esqueleto está no lugar, mas o docking não depende do esqueleto — depende da **geometria exata das cadeias laterais que formam o bolsão**. No modelo:

- as cadeias laterais do sítio podem estar em **rotâmeros do estado *apo*** (sem ligante), e a garganta pode estar "fechada";
- não há a **água** nem o **íon** estruturais;
- não há o **glicano**.

**Um RMSD global de 1 Å conviver com um sítio ativo inutilizável não é contradição — é a regra.** Métricas globais não garantem qualidade local, e o docking vive da qualidade local.

> **Decisão prática:** existe estrutura experimental de boa resolução, com ligante parecido com o seu, do organismo certo? **Use a experimental.** É o que faremos a seguir. Não existe? Use AlphaFold ou homologia — e **declare isso na metodologia** do seu artigo.

## B3. SWISS-MODEL (demonstração do docente, 2 min)

O **SWISS-MODEL** (<https://swissmodel.expasy.org>, do Instituto Suíço de Bioinformática) faz **modelagem por homologia**: você cola uma sequência, ele busca um molde com estrutura conhecida no PDB, alinha, constrói o modelo e devolve as métricas de qualidade:

- **GMQE** (0 a 1): qualidade **global** estimada;
- **QMEANDisCo** (0 a 1): qualidade **local**, resíduo a resíduo — abaixo de 0,6, desconfie do trecho;
- **gráfico de Ramachandran**: mostra se os ângulos da cadeia principal são geometricamente permitidos.

Você repetirá esse passo no **Exercício 1**, com a proteína da sua linha de pesquisa.


---
# BLOCO C — Docking com AutoDock Vina (33 min)

O **docking molecular** responde a duas perguntas sobre um par proteína–molécula pequena:

1. **Pose** — em que posição, orientação e conformação o ligante se encaixa no sítio?
2. **Score** — qual a estimativa de energia dessa interação, em kcal/mol (negativo = favorável)?

Vamos executar o fluxo de seis passos na ordem canônica:

```
1. Receptor : baixar → escolher UMA cadeia → remover águas e ligantes
              → adicionar hidrogênios → gerar .pdbqt com cargas
2. Ligante  : SMILES do PubChem → conformação 3D → protonação → torções → .pdbqt
3. Caixa    : centro e dimensões da região de busca
4. Rodar    : exhaustiveness, num_modes, seed
5. Validar  : redocking → RMSD
6. Analisar : a pose e as interações com os resíduos
```

## C1. Preparar o receptor — filtrando por **coluna**, não por texto

O formato `.pdb` é um texto de **colunas de largura fixa**. As posições importam:

```
ATOM    145  CA  SER A  203      12.345  67.890  10.111  1.00 18.52           C
|          |   |   |  |   |          |       |       |      |     |           |
col 1-6    |  13-16 | 22 23-26    31-38   39-46   47-54  55-60 61-66        77-78
tipo       7-11   resíduo                  coordenadas            B-factor  elemento
                  (18-20)  cadeia
```

Guarde dois números: **o nome do resíduo está nas colunas 18 a 20** e **a cadeia na coluna 22**.

**Por que não procurar simplesmente o texto "E20" na linha?** Porque isso encontraria também linhas de cabeçalho e — o problema sério — **as duas cópias do ligante**, uma na cadeia A e outra na cadeia B. O centro que calculássemos cairia no meio do caminho entre os dois sítios ativos, provavelmente no solvente. **O docking rodaria sem erro nenhum e o resultado seria lixo.** É a pergunta A.4 do Bloco A.


In [ ]:
# ===================================================================
# RECEPTOR: só a cadeia A, só a proteína (sem águas, glicanos ou ligantes)
# LIGANTE DE REFERÊNCIA: a donepezila (código E20) da cadeia A, apenas
# ===================================================================
# O 'awk' lê o arquivo linha a linha. 'substr($0, 22, 1)' pega 1 caractere
# a partir da coluna 22 — ou seja, o identificador da cadeia.
!awk '/^ATOM/ && substr($0,22,1)=="A"' 4EY7.pdb > receptor_limpo.pdb
!awk '/^HETATM/ && substr($0,18,3)=="E20" && substr($0,22,1)=="A"' 4EY7.pdb > ligante_cristal.pdb

# ----------------------- CONFERÊNCIA OBRIGATÓRIA -----------------------
n_receptor = sum(1 for _ in open("receptor_limpo.pdb"))
n_ligante  = sum(1 for _ in open("ligante_cristal.pdb"))
print(f"átomos do receptor (cadeia A)        : {n_receptor}")
print(f"átomos do ligante cristalográfico E20: {n_ligante}")

# A donepezila é C24H29NO3 -> 24 C + 1 N + 3 O = 28 átomos pesados.
# O cristal não traz hidrogênios, então esperamos exatamente 28 linhas.
if n_ligante == 28:
    print("\n✔ 28 átomos: correto. É UMA molécula de donepezila.")
elif n_ligante == 0:
    print("\n✖ ZERO átomos: o filtro não encontrou o E20. Confira se o 4EY7.pdb foi baixado.")
else:
    print(f"\n✖ {n_ligante} átomos — ERRADO. Se for ~56, você pegou as DUAS cadeias (A e B):")
    print("  o centro da caixa vai sair no meio do caminho entre os dois sítios ativos")
    print("  e o docking rodará sem reclamar. Volte e confira o filtro de cadeia.")

# Quantas cópias existem no arquivo inteiro? (demonstra o problema)
print("\nCópias de E20 no arquivo inteiro, por cadeia:")
!awk '/^HETATM/ && substr($0,18,3)=="E20" {print "   cadeia", substr($0,22,1)}' 4EY7.pdb | sort | uniq -c

### Hidrogênios e cargas: onde a maioria dos dockings ruins começa

O Vina precisa de um receptor em formato **PDBQT** = PDB **+ cargas parciais (Q) + tipos de átomo (T)**. Três coisas precisam estar certas:

| Opção do Open Babel | O que faz **de verdade** |
|---|---|
| `-xr` | Escreve o receptor como molécula **rígida**, sem árvore de torções. Obrigatório para receptor |
| `-p 7.4` | Adiciona hidrogênios conforme a protonação esperada em **pH 7,4** (o fisiológico) |
| `--partialcharge gasteiger` | Calcula as **cargas parciais** de Gasteiger. **Sem esta opção o arquivo sai com todas as cargas = 0,000** e a eletrostática do docking simplesmente desaparece |
| *(nada)* | Por **padrão** o Open Babel remove os hidrogênios **apolares** e mantém os **polares** — exatamente o modelo de "átomos unidos" que o Vina espera |

> ### Duas armadilhas de sintaxe que custam horas
>
> **1.** No Open Babel 3, `-xh` **não** significa "adicione só hidrogênios polares". Significa *preserve hydrogens*: mantém **todos** eles, inclusive os apolares que o Vina não quer.
>
> **2.** A opção global `-d` **apaga todos os hidrogênios**. Combinada com `-p 7.4`, desfaz o trabalho que acabou de ser feito: o receptor sai **sem nenhum átomo do tipo HD** (hidrogênio doador de ligação de hidrogênio) e os nitrogênios doadores são tipados como `NA`, isto é, como **aceptores**. O docking roda sem erro e perde todas as ligações de hidrogênio doadas pela proteína.
>
> A célula abaixo prepara o receptor das **duas** maneiras e conta os átomos de cada tipo, para você ver a diferença com os próprios olhos.


In [ ]:
# ===================================================================
# PREPARAÇÃO DO RECEPTOR — e uma comparação didática entre o certo e o errado
# ===================================================================

# >>> A FORMA CORRETA <<<
!obabel -ipdb receptor_limpo.pdb -opdbqt -O receptor.pdbqt -xr -p 7.4 --partialcharge gasteiger 2>/dev/null

# >>> A FORMA ERRADA (só para comparação; NÃO use) <<<
!obabel -ipdb receptor_limpo.pdb -opdbqt -O receptor_ERRADO.pdbqt -xr -xh -d --partialcharge gasteiger 2>/dev/null


def resumir_pdbqt(caminho):
    """Conta os tipos de átomo e as cargas não nulas de um arquivo PDBQT.

    No PDBQT, a última coluna é o TIPO de átomo do AutoDock e as colunas 71-78
    guardam a carga parcial. Tipos que interessam agora:
        HD = hidrogênio ligado a N/O -> DOADOR de ligação de hidrogênio
        NA, OA = nitrogênio/oxigênio ACEPTORES
        C, A = carbono alifático / aromático
    """
    tipos, cargas_nao_nulas, total = {}, 0, 0
    with open(caminho) as arquivo:
        for linha in arquivo:
            if linha.startswith(("ATOM", "HETATM")):
                total += 1
                tipo = linha.split()[-1]
                tipos[tipo] = tipos.get(tipo, 0) + 1
                try:
                    if abs(float(linha[70:76])) > 1e-6:
                        cargas_nao_nulas += 1
                except ValueError:
                    pass
    return total, tipos, cargas_nao_nulas


print(f"{'':28s} {'átomos':>8s} {'HD (doadores)':>15s} {'cargas ≠ 0':>12s}")
for rotulo, caminho in [("CORRETO  (-xr -p 7.4)", "receptor.pdbqt"),
                        ("ERRADO   (-xr -xh -d)", "receptor_ERRADO.pdbqt")]:
    total, tipos, cargas = resumir_pdbqt(caminho)
    print(f"{rotulo:28s} {total:8d} {tipos.get('HD', 0):15d} {cargas:12d}")

total, tipos, _ = resumir_pdbqt("receptor.pdbqt")
print("\nTipos de átomo no receptor correto:")
print("  " + "  ".join(f"{t}={n}" for t, n in sorted(tipos.items(), key=lambda p: -p[1])))

if tipos.get("HD", 0) == 0:
    print("\n✖ Receptor sem hidrogênios doadores — algo deu errado na preparação.")
else:
    print(f"\n✔ {tipos['HD']} hidrogênios polares (HD) presentes: o receptor pode DOAR ligações de hidrogênio.")
    print("  No arquivo errado esse número é ZERO — e o docking rodaria do mesmo jeito.")

## C2. Preparar o ligante — de um texto a uma molécula tridimensional

O **SMILES** (*Simplified Molecular Input Line Entry System*) é a notação em **texto** de uma molécula: você a copia direto da página do **PubChem**. O **RDKit** transforma esse texto em **coordenadas 3D** (porque um texto não tem geometria) e otimiza a geometria com um campo de força; o **Open Babel** adiciona as cargas e marca as **ligações rotacionáveis** que o Vina poderá girar durante a busca.

**Por que conferir a fórmula molecular?** Porque um SMILES colado errado produz uma molécula **parecida** que roda sem erro nenhum. A fórmula e a massa molar são a conferência barata deste passo.

Os ligantes de hoje são **inibidores reais da acetilcolinesterase usados na clínica**:

| Ligante | PubChem CID | Fórmula | Massa (g/mol) | Papel |
|---|---|---|---|---|
| **Donepezila** | 3152 | C₂₄H₂₉NO₃ | 379,5 | Inibidor de alta afinidade; é o **E20** do nosso cristal |
| **Galantamina** | 9651 | C₁₇H₂₁NO₃ | 287,4 | Inibidor mais fraco; molécula rígida, de origem natural |
| **Rivastigmina** | 77991 | C₁₄H₂₂N₂O₂ | 250,3 | **Carbamato** — a mesma família química dos inseticidas carbamatos |
| **Tacrina** | 1935 | C₁₃H₁₄N₂ | 198,3 | Pequena e plana; o primeiro aprovado para Alzheimer |

### De onde vêm os SMILES: do próprio PubChem, por uma API

Em vez de confiar num texto copiado à mão (que pode vir truncado ou de outra molécula), vamos **pedir o SMILES ao PubChem** através da sua **API** — endereços da web feitos para programas lerem, como fizemos com o UniProt na Aula 5.

E vamos fazer duas conferências, porque **nenhuma das duas sozinha basta**:

1. **A fórmula molecular devolvida é a esperada?** Protege contra um CID digitado errado — que traria, sem avisar, uma molécula completamente diferente.
2. **O RDKit consegue gerar uma geometria 3D?** Protege contra um SMILES **sintaticamente válido mas geometricamente impossível**. Um texto pode descrever uma molécula com a fórmula correta e uma conectividade que não fecha no espaço: a fórmula bate, e a molécula não existe. (Isto não é hipotético — foi exatamente o erro encontrado ao preparar este caderno.)


In [ ]:
# ===================================================================
# LIGANTES: do SMILES ao PDBQT
# ===================================================================
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, Lipinski

# Cada ligante: o CID do PubChem, a fórmula esperada e um SMILES de RESERVA
# (usado se a rede bloquear o PubChem). Reserva "None" = só pelo PubChem.
LIGANTES = {
    "donepezila":   {"cid": 3152,  "formula": "C24H29NO3",
                     "reserva": "COc1cc2c(cc1OC)C(=O)C(C2)CC1CCN(Cc2ccccc2)CC1"},
    "galantamina":  {"cid": 9651,  "formula": "C17H21NO3",  "reserva": None},
    "rivastigmina": {"cid": 77991, "formula": "C14H22N2O2",
                     "reserva": "CCN(C)C(=O)Oc1cccc(c1)C(C)N(C)C"},
    "tacrina":      {"cid": 1935,  "formula": "C13H14N2",
                     "reserva": "Nc1c2CCCCc2nc2ccccc12"},
}

PUBCHEM = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid"


def smiles_do_pubchem(cid, formula_esperada):
    """Pede o SMILES ao PubChem e CONFERE a fórmula molecular antes de aceitá-lo.

    A conferência da fórmula é o que protege contra um CID digitado errado:
    sem ela, um número trocado traria outra molécula sem nenhum aviso.
    """
    for propriedade in ("CanonicalSMILES", "ConnectivitySMILES", "IsomericSMILES"):
        try:
            url = f"{PUBCHEM}/{cid}/property/{propriedade},MolecularFormula/JSON"
            dados = requests.get(url, timeout=45).json()["PropertyTable"]["Properties"][0]
            smiles = dados.get(propriedade)
            formula = (dados.get("MolecularFormula") or "").replace("+", "").replace("-", "")
            if not smiles:
                continue
            if formula and formula != formula_esperada:
                print(f"    ✖ o CID {cid} devolveu {formula}, mas esperávamos {formula_esperada}"
                      f" — CID errado? Vou usar o SMILES de reserva.")
                return None
            return smiles
        except Exception:
            continue
    return None


def molecula_3d(smiles, semente=42):
    """Converte um SMILES em uma molécula com coordenadas 3D otimizadas.

    Devolve None se a geometria for IMPOSSÍVEL — o que acontece quando o SMILES
    é sintaticamente válido mas descreve uma conectividade que não fecha no espaço.
    """
    molecula = Chem.MolFromSmiles(smiles)
    if molecula is None:
        return None
    molecula = Chem.AddHs(molecula)          # hidrogênios explícitos: necessários para a geometria

    # ETKDG é o gerador de conformações do RDKit: usa regras de torção extraídas
    # de estruturas experimentais. A semente fixa torna o resultado reproduzível.
    parametros = AllChem.ETKDGv3()
    parametros.randomSeed = semente
    if AllChem.EmbedMolecule(molecula, parametros) != 0:
        parametros.useRandomCoords = True    # segunda tentativa, partindo de coordenadas aleatórias
        if AllChem.EmbedMolecule(molecula, parametros) != 0:
            return None                      # não há geometria possível para esta conectividade

    # MMFF94 é um campo de força para moléculas pequenas: relaxa ligações e ângulos.
    # Se a molécula não tiver parâmetros no MMFF, caímos no UFF, mais genérico.
    if AllChem.MMFFHasAllMoleculeParams(molecula):
        AllChem.MMFFOptimizeMolecule(molecula, maxIters=2000)
    else:
        AllChem.UFFOptimizeMolecule(molecula, maxIters=2000)
    return molecula


def preparar_ligante(nome, dados):
    """Obtém o SMILES, confere, gera a geometria 3D e escreve o .pdbqt do Vina."""
    print(f"  {nome}:")
    smiles = smiles_do_pubchem(dados["cid"], dados["formula"])
    origem = f"PubChem CID {dados['cid']}"
    if smiles is None:
        smiles = dados["reserva"]
        origem = "SMILES de reserva deste caderno"
        if smiles is None:
            print(f"    ✖ PubChem inacessível e não há reserva para {nome}.")
            print(f"       Abra https://pubchem.ncbi.nlm.nih.gov/compound/{dados['cid']},")
            print(f"       copie o campo 'SMILES' e cole em LIGANTES['{nome}']['reserva'].")
            return None
    print(f"    SMILES de: {origem}")

    molecula_plana = Chem.MolFromSmiles(smiles)
    if molecula_plana is None:
        print(f"    ✖ SMILES inválido para {nome}.")
        return None

    formula = rdMolDescriptors.CalcMolFormula(molecula_plana).replace("+", "").replace("-", "")
    massa = Descriptors.MolWt(molecula_plana)
    torcoes = Lipinski.NumRotatableBonds(molecula_plana)
    atomos_pesados = molecula_plana.GetNumHeavyAtoms()

    # CONFERÊNCIA 1 — a fórmula é a esperada?
    if formula != dados["formula"]:
        print(f"    ✖ fórmula {formula}, esperada {dados['formula']}. Não vou usar esta molécula.")
        return None

    # CONFERÊNCIA 2 — existe geometria 3D possível?
    molecula = molecula_3d(smiles)
    if molecula is None:
        print(f"    ✖ fórmula correta ({formula}), mas o RDKit NÃO conseguiu gerar uma")
        print("       geometria 3D: a conectividade descrita não fecha no espaço.")
        print("       Pegue o SMILES diretamente da página do PubChem.")
        return None

    Chem.MolToMolFile(molecula, f"{nome}.sdf")
    # O Open Babel converte para PDBQT: calcula as cargas e monta a árvore de torções.
    # (Por padrão ele remove os hidrogênios apolares, como o Vina espera.)
    os.system(f"obabel -isdf {nome}.sdf -opdbqt -O {nome}.pdbqt --partialcharge gasteiger 2>/dev/null")
    n_branch = sum(1 for linha in open(f"{nome}.pdbqt") if linha.startswith("BRANCH"))

    print(f"    ✔ {formula}  massa={massa:.1f} g/mol  átomos pesados={atomos_pesados}  "
          f"torções={torcoes}  (BRANCH no PDBQT: {n_branch})")
    return {"nome": nome, "formula": formula, "massa": massa, "smiles": smiles,
            "atomos_pesados": atomos_pesados, "torcoes": torcoes, "branches": n_branch}


print("Obtendo e preparando os ligantes:\n")
info_ligantes = {}
for nome, dados in LIGANTES.items():
    resultado = preparar_ligante(nome, dados)
    if resultado:
        info_ligantes[nome] = resultado

print(f"\n{len(info_ligantes)} de {len(LIGANTES)} ligantes prontos:",
      ", ".join(info_ligantes))
if "donepezila" not in info_ligantes:
    print("\n✖ A donepezila é indispensável (é o ligante do cristal). Resolva antes de seguir.")

**Pergunta C.1** — Todas as fórmulas bateram? Algum ligante foi recusado por uma das duas conferências? Qual ligante tem mais torções? Pense no tamanho do espaço de busca: além das 3 translações e 3 rotações, cada torção acrescenta **uma dimensão**. A donepezila, com 6 torções, define um espaço de **12 dimensões** — por isso a busca é **estocástica** (Monte Carlo), e não exaustiva.

## C3. A caixa de busca — o parâmetro que mais erra

A **caixa** (*grid box*) define **onde** o programa procura.

- **Docking dirigido** (*site-specific*): caixa pequena, de 20 a 25 Å de lado, centrada no sítio ativo conhecido. Mais rápido e mais confiável — é o nosso caso.
- **Docking cego** (*blind*): caixa cobrindo a proteína inteira, quando o sítio é desconhecido. Muito mais ruidoso.

**Como achar o centro:** o cristal já traz um ligante no sítio, então basta o **centro geométrico desse ligante** — de **uma única cadeia**, como garantimos em C1.

**Regra prática da aresta:** a maior dimensão do ligante **mais** cerca de 8 Å, para que ele possa girar livremente dentro da caixa.


In [ ]:
# ===================================================================
# CENTRO E TAMANHO DA CAIXA
# ===================================================================
# Lemos as coordenadas das COLUNAS FIXAS do PDB: x em 31-38, y em 39-46, z em 47-54.
# (Em Python as posições começam em 0, por isso [30:38], [38:46], [46:54].)
coordenadas = np.array([[float(linha[30:38]), float(linha[38:46]), float(linha[46:54])]
                        for linha in open("ligante_cristal.pdb") if linha.startswith("HETATM")])

print(f"átomos lidos: {len(coordenadas)}   (DEVE ser 28 — se for ~56, pare: são as duas cadeias)")

centro = coordenadas.mean(axis=0)
extensao = coordenadas.max(axis=0) - coordenadas.min(axis=0)
CX, CY, CZ = centro

print(f"\ncentro da caixa : x={CX:.3f}  y={CY:.3f}  z={CZ:.3f}")
print(f"extensão do ligante (Å): {np.round(extensao, 1)}  -> maior dimensão = {extensao.max():.1f} Å")

TAMANHO = 22            # aresta da caixa, em ångström
print(f"\naresta escolhida: {TAMANHO} Å")
print(f"folga sobre a maior dimensão do ligante: {TAMANHO - extensao.max():.1f} Å "
      f"({'suficiente' if TAMANHO - extensao.max() >= 6 else 'POUCA — aumente a caixa'})")

# Diagnóstico extra: o centro está mesmo DENTRO da proteína, cercado de resíduos?
# Se tivéssemos pegado as duas cadeias, o centro cairia no vazio e este número seria baixo.
atomos_receptor = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                            for l in open("receptor_limpo.pdb") if l.startswith("ATOM")])
distancias = np.linalg.norm(atomos_receptor - centro, axis=1)
vizinhos = int((distancias < 10).sum())
print(f"\nátomos do receptor a menos de 10 Å do centro: {vizinhos}")
print("  ✔ centro enterrado na proteína (esperado: algumas centenas)" if vizinhos > 150
      else "  ✖ centro em região vazia — provavelmente o filtro de cadeia falhou")

## C4. Rodar o docking

Os parâmetros que você vai ajustar:

| Parâmetro | O que faz |
|---|---|
| `--exhaustiveness` | Quantas buscas **independentes** são feitas. Padrão 8; 16 ou 32 dão resultados mais reprodutíveis e levam mais tempo |
| `--num_modes` | Quantas poses retornar (padrão 9) |
| `--energy_range` | Diferença máxima de energia, em kcal/mol, entre a melhor pose e as demais reportadas |
| `--seed` | A **semente** do gerador de números aleatórios. **Docking é estocástico:** fixá-la é o que torna o seu resultado reproduzível por outra pessoa — e por você mesmo, em seis meses |

**Tempo esperado:** cerca de **1 minuto por ligante** nesta configuração, na máquina gratuita do Colab.


In [ ]:
# ===================================================================
# DOCKING — a donepezila primeiro (é a que tem pose cristalográfica para comparar)
# ===================================================================
EXHAUSTIVENESS = 16
SEMENTE = 42


def rodar_vina(ligante_pdbqt, saida_pdbqt, log_txt,
               exhaustiveness=EXHAUSTIVENESS, semente=SEMENTE):
    """Executa o AutoDock Vina e devolve a lista de afinidades (kcal/mol) das poses."""
    comando = (
        f"./vina --receptor receptor.pdbqt --ligand {ligante_pdbqt} "
        f"--center_x {CX:.3f} --center_y {CY:.3f} --center_z {CZ:.3f} "
        f"--size_x {TAMANHO} --size_y {TAMANHO} --size_z {TAMANHO} "
        f"--exhaustiveness {exhaustiveness} --num_modes 9 --seed {semente} "
        f"--out {saida_pdbqt} > {log_txt} 2>&1"
    )
    inicio = time.time()
    os.system(comando)
    duracao = time.time() - inicio

    # A tabela de saída do Vina tem linhas como:  "   1       -12.4      0.000      0.000"
    afinidades = []
    for linha in open(log_txt):
        partes = linha.split()
        if len(partes) == 4 and partes[0].isdigit():
            afinidades.append(float(partes[1]))
    return afinidades, duracao


print(f"Rodando o Vina (exhaustiveness={EXHAUSTIVENESS}, seed={SEMENTE})...")
afinidades, duracao = rodar_vina("donepezila.pdbqt", "poses_donepezila.pdbqt", "log_donepezila.txt")

if not afinidades:
    print("✖ O Vina não produziu poses. Veja a saída completa:")
    print(open("log_donepezila.txt").read()[-2000:])
else:
    print(f"✔ {len(afinidades)} poses em {duracao:.0f} segundos\n")
    tabela_poses = pd.DataFrame({"pose": range(1, len(afinidades) + 1),
                                 "afinidade (kcal/mol)": afinidades})
    tabela_poses["diferença para a melhor"] = (tabela_poses["afinidade (kcal/mol)"]
                                               - afinidades[0]).round(2)
    print(tabela_poses.to_string(index=False))
    print(f"\nMelhor afinidade (pose 1): {afinidades[0]:.2f} kcal/mol")
    print("Lembre: valores NEGATIVOS são favoráveis, e este número é uma ESTIMATIVA, não uma medida.")

## C5. Validar: redocking e RMSD

**Esta é a parte que todo mundo pula e o avaliador cobra.**

O **redocking** é o experimento de controle: pegamos a estrutura cristalográfica que **já contém** um ligante, retiramos esse ligante (foi o que fizemos em C1) e pedimos ao programa que o **recoloque**. Depois comparamos a pose prevista com a pose real.

A métrica é o **RMSD** entre os átomos pesados:

| RMSD | Leitura |
|---|---|
| **< 2,0 Å** | O protocolo reproduziu a realidade. **Protocolo validado** |
| 2 – 3 Å | Parcialmente correto: geralmente a orientação certa, os detalhes errados |
| > 3 Å | O protocolo não serve para esse alvo. Revise preparação, caixa e protonação |

Usamos o **`obrms`** (do Open Babel), que faz o **pareamento de átomos por grafo** antes de calcular o RMSD: ele reconhece que o átomo nº 7 de um arquivo corresponde ao nº 12 do outro, resolvendo o isomorfismo entre as duas moléculas. Um RMSD "ingênuo", que comparasse os átomos na ordem em que aparecem no arquivo, daria um número errado.

> ### A armadilha lógica do redocking
> Ele valida o **protocolo**, não o **score**. Um RMSD de 0,3 Å prova que o seu pipeline encontra a pose certa quando ela existe; **não** prova que o número em kcal/mol corresponde à afinidade medida em laboratório. São duas afirmações diferentes, e confundi-las é o erro mais comum em trabalhos de disciplina.


In [ ]:
# ===================================================================
# REDOCKING: a pose prevista reproduz a pose cristalográfica?
# ===================================================================
# Extrai a pose 1 (a melhor) do arquivo de poses. "-f 1 -l 1" = da primeira à primeira.
!obabel -ipdbqt poses_donepezila.pdbqt -opdb -O pose_melhor.pdb -f 1 -l 1 2>/dev/null

# obrms compara duas moléculas pareando os átomos por grafo
saida_obrms = subprocess.run("obrms ligante_cristal.pdb pose_melhor.pdb",
                             shell=True, capture_output=True, text=True).stdout.strip()
print("saída do obrms:", saida_obrms or "(vazia)")

try:
    rmsd_redocking = float(saida_obrms.split()[-1])
except (ValueError, IndexError):
    rmsd_redocking = float("nan")
    print("\n✖ Não consegui ler um número na saída do obrms. Confira se os dois arquivos existem.")

if np.isinf(rmsd_redocking):
    print("""
>>> RMSD = infinito. Isto NÃO é um RMSD grande: é o obrms dizendo que não
    conseguiu PAREAR os átomos dos dois arquivos, porque eles descrevem
    moléculas DIFERENTES (número ou tipo de átomos incompatível).
    Confira se o ligante que você dockou é o mesmo do cristal.""")
elif rmsd_redocking < 2.0:
    print("✔ PROTOCOLO VALIDADO (RMSD < 2,0 Å).")
    print("  Podemos confiar neste protocolo para testar moléculas novas no MESMO alvo.")
    print("  Atenção: isto valida a POSE, não o SCORE.")
elif rmsd_redocking < 3.0 and not np.isnan(rmsd_redocking):
    print("~ Parcialmente correto. Orientação provavelmente certa, detalhes errados.")
elif not np.isnan(rmsd_redocking) and not np.isinf(rmsd_redocking):
    print("✖ RMSD alto. Revise, nesta ordem:")
    print("   1. o centro da caixa (C3) — o número de átomos do ligante era 28?")
    print("   2. as cargas do receptor (C1) — havia átomos HD?")
    print("   3. o tamanho da caixa — a folga era suficiente?")

## C6. Analisar: quem toca o ligante?

Um score sem um mapa de interações é um número solto. Vamos descobrir **quais resíduos da proteína ficam a menos de 4 Å** da pose prevista — a distância típica de um contato relevante — e comparar com a pose cristalográfica.

Se os dois conjuntos de resíduos coincidirem, a conclusão é forte: **o docking reencontrou o mesmo bolsão, com os mesmos contatos.**


In [ ]:
# ===================================================================
# RESÍDUOS EM CONTATO COM O LIGANTE (dentro de 4 Å)
# ===================================================================
RAIO_CONTATO = 4.0

# (reimportamos aqui para que este bloco funcione mesmo que você pule o Bloco B)
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
leitor = PDBParser(QUIET=True)

estrutura_receptor = leitor.get_structure("rec", "receptor_limpo.pdb")


def residuos_proximos(arquivo_ligante, raio=RAIO_CONTATO):
    """Lista os resíduos do receptor com algum átomo a menos de 'raio' de algum átomo do ligante."""
    pontos = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                       for l in open(arquivo_ligante)
                       if l.startswith(("ATOM", "HETATM"))])
    achados = {}
    for residuo in estrutura_receptor[0]["A"]:
        if not is_aa(residuo, standard=False):
            continue
        coords = np.array([atomo.coord for atomo in residuo])
        # distância de cada átomo do resíduo a cada átomo do ligante
        menor = np.linalg.norm(coords[:, None, :] - pontos[None, :, :], axis=2).min()
        if menor < raio:
            achados[residuo.id[1]] = (f"{residuo.get_resname()}{residuo.id[1]}", round(float(menor), 2))
    return achados


contatos_cristal = residuos_proximos("ligante_cristal.pdb")
contatos_pose    = residuos_proximos("pose_melhor.pdb")

comuns = sorted(set(contatos_cristal) & set(contatos_pose))
so_cristal = sorted(set(contatos_cristal) - set(contatos_pose))
so_pose    = sorted(set(contatos_pose) - set(contatos_cristal))

print(f"Resíduos a menos de {RAIO_CONTATO} Å do ligante:")
print(f"  na pose CRISTALOGRÁFICA : {len(contatos_cristal)}")
print(f"  na pose PREVISTA        : {len(contatos_pose)}")
print(f"  em COMUM                : {len(comuns)}  "
      f"({100*len(comuns)/max(len(contatos_cristal),1):.0f}% dos contatos do cristal)")

print("\nResíduos em contato nas DUAS poses:")
print("  " + ", ".join(contatos_cristal[r][0] for r in comuns))
if so_cristal:
    print("\nSó no cristal :", ", ".join(contatos_cristal[r][0] for r in so_cristal))
if so_pose:
    print("Só na previsão:", ", ".join(contatos_pose[r][0] for r in so_pose))

# Os aromáticos da "garganta" e a tríade catalítica
AROMATICOS = ("TRP", "TYR", "PHE")
n_aromaticos = sum(1 for r in contatos_pose if contatos_pose[r][0][:3] in AROMATICOS)
print(f"\nDesses contatos previstos, {n_aromaticos} são com resíduos AROMÁTICOS "
      f"(TRP/TYR/PHE) — a assinatura da garganta aromática da AChE.")

for alvo, apelido in [(86, "TRP86, fundo da garganta"), (286, "TRP286, sítio periférico"),
                      (203, "SER203, serina catalítica")]:
    if alvo in contatos_pose:
        print(f"  ✔ {apelido}: em contato, a {contatos_pose[alvo][1]} Å")
    else:
        print(f"  – {apelido}: fora do raio de {RAIO_CONTATO} Å")

**Pergunta C.2** — Que fração dos contatos cristalográficos o docking reencontrou? A donepezila toca **ao mesmo tempo** o TRP86 (fundo) e o TRP286 (entrada)? Isso confirma a hipótese do Bloco A sobre por que ela é um inibidor potente?

## C7. Triagem comparativa — e o viés que arruína conclusões ingênuas

Agora dockamos todos os ligantes no mesmo sítio e comparamos. **Cuidado com a leitura:** o score do Vina tem um **viés conhecido a favor de moléculas maiores**, porque mais átomos significam mais termos favoráveis de van der Waals. Comparar o score bruto de uma molécula de 28 átomos pesados com uma de 15 é **comparar coisas diferentes**.

A correção usual é a ***ligand efficiency*** (eficiência do ligante):

$$ \text{LE} = \frac{-\Delta G}{\text{n}^\circ \text{ de átomos pesados}} $$

Ela responde a uma pergunta mais justa: **quanta afinidade cada átomo está "pagando"?**


In [ ]:
# ===================================================================
# TRIAGEM: os três ligantes no mesmo sítio
# ===================================================================
print(f"Dockando {len(info_ligantes)} ligantes — cerca de 1 minuto cada, tenha paciência.\n")
resultados = []
for nome in info_ligantes:
    afins, dur = rodar_vina(f"{nome}.pdbqt", f"poses_{nome}.pdbqt", f"log_{nome}.txt")
    if not afins:
        print(f"✖ {nome}: o Vina não produziu poses (veja log_{nome}.txt)")
        continue
    info = info_ligantes[nome]
    resultados.append({
        "ligante": nome,
        "átomos pesados": info["atomos_pesados"],
        "torções": info["torcoes"],
        "afinidade (kcal/mol)": afins[0],
        "ligand efficiency": round(-afins[0] / info["atomos_pesados"], 3),
        "tempo (s)": round(dur),
    })
    print(f"  {nome:12s} {afins[0]:7.2f} kcal/mol  ({dur:.0f} s)")

triagem = pd.DataFrame(resultados)
print("\n=== Tabela da triagem ===")
print(triagem.sort_values("afinidade (kcal/mol)").to_string(index=False))

# Gráfico: as duas leituras possíveis, lado a lado
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.2))
ordem_score = triagem.sort_values("afinidade (kcal/mol)")
eixos[0].barh(ordem_score["ligante"], -ordem_score["afinidade (kcal/mol)"], color=COR_B)
eixos[0].set_xlabel("−afinidade (kcal/mol) — maior é melhor")
eixos[0].set_title("Score bruto: favorece a molécula GRANDE", loc="left", color=TINTA)

ordem_le = triagem.sort_values("ligand efficiency")
eixos[1].barh(ordem_le["ligante"], ordem_le["ligand efficiency"], color=COR_A)
eixos[1].set_xlabel("ligand efficiency (kcal/mol por átomo pesado)")
eixos[1].set_title("Normalizado pelo tamanho: outra ordem", loc="left", color=TINTA)
for eixo in eixos:
    eixo.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

print("Pergunta C.3 — As duas ordens são a mesma? Qual delas você reportaria num artigo, e por quê?")

**Pergunta C.3** — As duas ordens coincidem? Qual você reportaria?

**Pergunta C.4** — No **Exercício 3** você vai comparar essa ordem com os valores de **IC50** publicados (busque no **ChEMBL**, <https://www.ebi.ac.uk/chembl>). Antes de olhar: você **espera** que a ordem bata? Lembre-se de que a correlação entre score do Vina e afinidade medida é **modesta** — docking **classifica e prioriza** candidatos, não mede afinidade.

## C8. (Opcional, mas instrutivo) Reproduzindo o erro de propósito

Esta célula faz **exatamente o que não se deve fazer**: extrai o ligante cristalográfico **sem filtrar a cadeia**. É o Exercício 4 do roteiro, e serve para você ver, com os próprios olhos, **um resultado errado que não gera nenhuma mensagem de erro**.


In [ ]:
# ===================================================================
# DEMONSTRAÇÃO DO ERRO — não copie isto para o seu trabalho!
# ===================================================================
# Note a AUSÊNCIA do filtro de cadeia:
!awk '/^HETATM/ && substr($0,18,3)=="E20"' 4EY7.pdb > ligante_ERRADO.pdb

coords_erradas = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                           for l in open("ligante_ERRADO.pdb") if l.startswith("HETATM")])
centro_errado = coords_erradas.mean(axis=0)

print(f"átomos 'do ligante'      : {len(coords_erradas)}   (eram 28 com o filtro correto)")
print(f"centro CORRETO           : {np.round(centro, 2)}")
print(f"centro ERRADO            : {np.round(centro_errado, 2)}")
print(f"distância entre os dois  : {np.linalg.norm(centro - centro_errado):.1f} Å")

d_erradas = np.linalg.norm(atomos_receptor - centro_errado, axis=1)
print(f"\nátomos do receptor a menos de 10 Å do centro errado: {int((d_erradas < 10).sum())}")
print(f"  (no centro correto eram {vizinhos})")

# E agora o ponto central: o docking roda, e produz um número.
print("\nRodando o Vina com a caixa no lugar errado...")
CX_bom, CY_bom, CZ_bom = CX, CY, CZ          # guardamos os valores corretos
CX, CY, CZ = centro_errado                    # usamos os errados só nesta célula
afins_erradas, _ = rodar_vina("donepezila.pdbqt", "poses_ERRADAS.pdbqt", "log_ERRADO.txt")
CX, CY, CZ = CX_bom, CY_bom, CZ_bom           # restauramos imediatamente

if afins_erradas:
    # (usamos os.system em vez de "!" porque estamos dentro de um bloco indentado)
    os.system("obabel -ipdbqt poses_ERRADAS.pdbqt -opdb -O pose_ERRADA.pdb -f 1 -l 1 2>/dev/null")
    rmsd_errado = float(subprocess.run("obrms ligante_cristal.pdb pose_ERRADA.pdb", shell=True,
                                       capture_output=True, text=True).stdout.split()[-1])
    print(f"\n{'':24s} {'afinidade':>12s} {'RMSD redocking':>16s}")
    print(f"{'protocolo CORRETO':24s} {afinidades[0]:12.2f} {rmsd_redocking:15.2f} Å")
    print(f"{'protocolo ERRADO':24s} {afins_erradas[0]:12.2f} {rmsd_errado:15.2f} Å")
    print("\n>>> Repare: o protocolo errado produziu uma afinidade de aparência PERFEITAMENTE")
    print(">>> PLAUSÍVEL. Nenhum aviso, nenhum erro. A ÚNICA coisa que denunciou o problema")
    print(">>> foi o RMSD do redocking — e, antes dele, a contagem de 28 átomos.")
    print("\nÉ por isso que a validação não é opcional.")
else:
    print("O Vina não produziu poses nesta configuração — o que também é um sinal de problema.")

---
# BLOCO D — Dinâmica molecular: lendo uma trajetória (15 min)

O docking entrega uma **fotografia**. A proteína real é um **filme**: ela vibra, respira, abre e fecha cavidades. A **dinâmica molecular (MD)** calcula esse filme resolvendo **F = ma** para cada átomo, em passos de cerca de 2 femtossegundos, repetidamente, milhões de vezes.

Uma simulação de verdade não cabe em 15 minutos de aula: o complexo AChE–donepezila solvatado tem cerca de **100 mil átomos**, e 100 ns levariam **horas de GPU**. Então vamos fazer o que de fato é a habilidade transferível: **analisar uma trajetória pronta e interpretar os gráficos**.

### O conjunto de dados

Usaremos a trajetória de referência que **acompanha a própria biblioteca MDAnalysis**: a **adenilato quinase (AdK)** de *E. coli* (estrutura de partida **4AKE**). É pública, citável e já vem instalada — ninguém precisa baixar nada.

**Por que a AdK e não a AChE?** Porque ela faz, em cerca de 1 ns, algo que a AChE não faria: **abre os seus domínios de forma visível**. É um dos exemplos didáticos mais usados do mundo justamente porque mostra, em poucos quadros, a diferença entre *"o sistema não convergiu"* e *"está acontecendo uma transição conformacional real"* — a interpretação mais difícil e mais cobrada em MD.


In [ ]:
# ===================================================================
# CARREGANDO A TRAJETÓRIA
# ===================================================================
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
from MDAnalysis.tests.datafiles import PSF, DCD   # a trajetória de exemplo da própria biblioteca

# Um "Universe" junta a TOPOLOGIA (quais átomos existem, como se ligam: o arquivo PSF)
# com a TRAJETÓRIA (as coordenadas de cada átomo em cada instante: o arquivo DCD).
u = mda.Universe(PSF, DCD)

print(f"átomos    : {len(u.atoms)}")
print(f"resíduos  : {len(u.residues)}")
print(f"quadros   : {len(u.trajectory)}")
print(f"tempo total: {u.trajectory.totaltime:.0f} ps  (≈ {u.trajectory.totaltime/1000:.2f} ns)")
print("\nEsperado: 3341 átomos · 214 resíduos · 98 quadros · 97 ps")

## D1. RMSD × tempo — a estrutura se afastou quanto da inicial?

O **RMSD** em função do tempo é o primeiro gráfico de qualquer análise de MD. A leitura clássica:

- **sobe e estabiliza num platô** → o sistema equilibrou. É o que se quer ver;
- **sobe continuamente** → não convergiu... **ou** está acontecendo algo real.

Vamos ver qual dos dois é o nosso caso.


In [ ]:
# rms.RMSD compara cada quadro com um quadro de referência (aqui, o primeiro).
# "backbone" = os átomos da cadeia principal (N, CA, C, O): o esqueleto da proteína.
analise_rmsd = rms.RMSD(u, u, select="backbone", ref_frame=0).run()
quadro = analise_rmsd.results.rmsd[:, 1]
valores_rmsd = analise_rmsd.results.rmsd[:, 2]

fig, eixo = plt.subplots(figsize=(9, 4))
eixo.plot(quadro, valores_rmsd, lw=1.8, color=COR_A)
eixo.set_xlabel("quadro da trajetória")
eixo.set_ylabel("RMSD do esqueleto (Å)")
eixo.set_title("RMSD em relação ao quadro inicial", loc="left", color=TINTA)
plt.tight_layout(); plt.show()

print(f"RMSD inicial : {valores_rmsd[0]:.2f} Å   (zero, por definição: é a referência)")
print(f"RMSD final   : {valores_rmsd[-1]:.2f} Å")
print(f"RMSD máximo  : {valores_rmsd.max():.2f} Å")
print("\nEsperado: sobe continuamente até cerca de 6,8 Å, SEM atingir platô.")
print("""
Interprete — e resista à conclusão fácil.
Num complexo proteína–ligante equilibrado, isto significaria "não convergiu,
simulação inútil". Aqui significa outra coisa. Como distinguir os dois casos?
Com a métrica seguinte.""")

## D2. Raio de giro — a proteína está abrindo ou desenovelando?

O **raio de giro (Rg)** mede a **compactação global**: a distância média dos átomos ao centro de massa. É a métrica que desempata a ambiguidade do gráfico anterior.

- Rg **estável** → a proteína mantém o enovelamento;
- Rg **crescendo** → está abrindo **ou** desenovelando. (E são coisas muito diferentes!)


In [ ]:
proteina = u.select_atoms("protein")

# Percorremos a trajetória quadro a quadro; em cada um, calculamos o raio de giro.
raio_giro = np.array([proteina.radius_of_gyration() for _ in u.trajectory])

fig, eixo = plt.subplots(figsize=(9, 4))
eixo.plot(raio_giro, lw=1.8, color=COR_B)
eixo.set_xlabel("quadro da trajetória")
eixo.set_ylabel("raio de giro (Å)")
eixo.set_title("Compactação global", loc="left", color=TINTA)
plt.tight_layout(); plt.show()

print(f"Rg inicial : {raio_giro[0]:.2f} Å")
print(f"Rg final   : {raio_giro[-1]:.2f} Å")
print(f"variação   : {raio_giro[-1] - raio_giro[0]:+.2f} Å")
print("\nEsperado: cresce de cerca de 16,7 Å para 19,6 Å.")
print("""
Interprete: a proteína ficou MAIS ABERTA (cerca de +3 Å de raio de giro),
mas manteve a estrutura secundária. É ABERTURA DE DOMÍNIOS — a função
biológica da adenilato quinase — e não desenovelamento. Se fosse
desenovelamento, o Rg dispararia e a estrutura secundária colapsaria.

>>> A leitura correta de uma trajetória NUNCA vem de um gráfico só. <<<""")

## D3. RMSF por resíduo — quem se move?

O **RMSF** (*Root Mean Square Fluctuation*) mede, para **cada resíduo**, quanto ele oscila em torno da sua posição média ao longo do tempo. É o retrato da **flexibilidade**.

> ### Uma nota metodológica obrigatória
> O **alinhamento da trajetória vem antes** do RMSF. Sem alinhar, você mede o movimento da proteína **como um corpo rígido pela caixa de simulação** — a translação e a rotação globais — e **não** a flexibilidade interna. Os números saem grandes, plausíveis, e não querem dizer nada. É um erro silencioso muito comum.


In [ ]:
# AlignTraj remove a translação e a rotação globais, sobrepondo cada quadro ao primeiro.
# SEM este passo, o RMSF mediria o passeio da proteína pela caixa, não sua flexibilidade.
align.AlignTraj(u, u, select="protein and name CA", in_memory=True).run()

carbonos_alfa_md = u.select_atoms("protein and name CA")
analise_rmsf = rms.RMSF(carbonos_alfa_md).run()
rmsf = analise_rmsf.results.rmsf

fig, eixo = plt.subplots(figsize=(11, 4))
eixo.plot(carbonos_alfa_md.resids, rmsf, lw=1.5, color="#1D5A33")
eixo.axhline(rmsf.mean(), ls="--", lw=1, color=TINTA_SUAVE)
eixo.set_xlabel("número do resíduo")
eixo.set_ylabel("RMSF (Å)")
eixo.set_title("Flexibilidade por resíduo (tracejado = média)", loc="left", color=TINTA)
plt.tight_layout(); plt.show()

print(f"RMSF médio : {rmsf.mean():.2f} Å")
print(f"RMSF máximo: {rmsf.max():.2f} Å, no resíduo {carbonos_alfa_md.resids[rmsf.argmax()]}")
print("\nEsperado: média em torno de 1,9 Å e máximo de cerca de 5,7 Å perto do resíduo 149.")

# Os cinco resíduos mais móveis e os cinco mais rígidos
ordem = np.argsort(rmsf)
print("\nmais RÍGIDOS :", ", ".join(f"{carbonos_alfa_md.resids[i]} ({rmsf[i]:.1f} Å)" for i in ordem[:5]))
print("mais MÓVEIS  :", ", ".join(f"{carbonos_alfa_md.resids[i]} ({rmsf[i]:.1f} Å)" for i in ordem[-5:]))
print("""
Interprete: os picos não estão distribuídos ao acaso — concentram-se nos
domínios móveis (as "tampas" LID e NMP da adenilato quinase), enquanto o
núcleo permanece rígido. É exatamente esse tipo de gráfico que você usaria
para argumentar que o SEU sítio ativo é estável enquanto a periferia respira.""")

### Para rodar a sua própria simulação depois

A sequência mínima no **GROMACS** (referência; não precisa executar hoje):

```bash
gmx pdb2gmx  -f proteina.pdb -o proc.gro -water tip3p      # topologia + hidrogênios
gmx editconf -f proc.gro -o box.gro -c -d 1.0 -bt cubic    # caixa
gmx solvate  -cp box.gro -cs spc216.gro -o solv.gro -p topol.top
gmx grompp   -f ions.mdp -c solv.gro -p topol.top -o ions.tpr
gmx genion   -s ions.tpr -o solv_ions.gro -p topol.top -neutral -conc 0.15
gmx grompp -f minim.mdp ... && gmx mdrun -v -deffnm em     # minimização
gmx grompp -f nvt.mdp   ... && gmx mdrun -deffnm nvt       # equilíbrio NVT (310 K)
gmx grompp -f npt.mdp   ... && gmx mdrun -deffnm npt       # equilíbrio NPT (1 bar)
gmx grompp -f md.mdp    ... && gmx mdrun -deffnm md        # produção
```

Para um complexo **com ligante**, o passo extra — e mais delicado — é **parametrizar o ligante** (CGenFF para CHARMM, ACPYPE/GAFF para AMBER). Os campos de força foram ajustados para aminoácidos, ácidos nucleicos e água; **o seu ligante não estava na lista**, e é aí que mora a maior parte dos erros em MD de complexos.

> **E o mais importante:** uma MD de 100 ns **não prova afinidade**. Ela testa a **estabilidade** da pose que o docking propôs. É assim — e só assim — que se deve escrever no artigo.


In [ ]:
# ===================================================================
# RESUMO DA SUA EXECUÇÃO — copie estes números para o caderno de laboratório
# ===================================================================
import platform

print("=" * 68)
print("AULA 7 — RESUMO DA PRÁTICA")
print("=" * 68)
print(f"Python           : {platform.python_version()}")
print(f"AutoDock Vina    : {VINA_VERSAO}")
print(f"Open Babel       : via {origem_obabel}")
print()
print(f"Receptor         : 4EY7 cadeia A — {n_receptor} átomos, raios X 2,35 Å, R-free 0,211")
print(f"Ligante de ref.  : E20 (donepezila) cadeia A — {n_ligante} átomos pesados")
print(f"Caixa            : centro ({CX:.2f}, {CY:.2f}, {CZ:.2f}), aresta {TAMANHO} Å")
print(f"Busca            : exhaustiveness={EXHAUSTIVENESS}, num_modes=9, seed={SEMENTE}")
print(f"Melhor afinidade : {afinidades[0]:.2f} kcal/mol")
print(f"RMSD redocking   : {rmsd_redocking:.2f} Å  -> {'VALIDADO' if rmsd_redocking < 2 else 'NÃO validado'}")
print(f"Contatos < 4 Å   : {len(contatos_pose)} resíduos, {len(comuns)} coincidem com o cristal")
print()
print(f"AlphaFold P22303 : RMSD global {rmsd:.2f} Å, deslocamento de numeração {deslocamento:+d}")
print(f"Trajetória AdK   : RMSD final {valores_rmsd[-1]:.2f} Å, Rg {raio_giro[0]:.1f} -> {raio_giro[-1]:.1f} Å, "
      f"RMSF médio {rmsf.mean():.2f} Å")
print("=" * 68)

# Salva a tabela da triagem para você anexar ao relatório
triagem.to_csv("triagem_ache.csv", index=False)
print("\nTabela da triagem salva em 'triagem_ache.csv'.")
try:
    from google.colab import files
    files.download("triagem_ache.csv")
except ImportError:
    pass   # fora do Colab, o arquivo fica na pasta atual

---
# Conexão com a teoria

| Célula deste caderno | Conceito da aula |
|---|---|
| Ficha da 4EY7 no Bloco A | Resolução, R-free e **número de cadeias** |
| `substr($0,22,1)=="A"` | Unidade assimétrica × unidade biológica; colunas de largura fixa do PDB |
| Conferência dos 28 átomos | A classe de erros que **não gera mensagem de erro** |
| `-p 7.4` e `--partialcharge gasteiger` | Protonação em pH fisiológico e eletrostática |
| Comparação HD = 509 × HD = 0 | Preparação ruim é a causa mais comum de docking ruim |
| `Chem.MolFromSmiles` + `EmbedMolecule` | Do texto SMILES à geometria 3D |
| Contagem de `BRANCH` | Ligações rotacionáveis = dimensões do espaço de busca |
| Centro geométrico do ligante cristalográfico | Docking dirigido × docking cego |
| `--seed` e `--exhaustiveness` | Docking é **estocástico**; reprodutibilidade |
| `obrms` e o RMSD < 2 Å | **Validação por redocking** — valida a pose, não o score |
| Resíduos a menos de 4 Å | Mapa de interações; a garganta aromática da AChE |
| *Ligand efficiency* | O viés da função de pontuação a favor de moléculas grandes |
| Alinhamento de sequência antes de sobrepor | Numeração do cristal × numeração do UniProt (peptídeo-sinal) |
| pLDDT × desvio real | O AlphaFold **sabe onde não sabe**; qualidade global ≠ qualidade local |
| RMSD sem platô + Rg crescente | Convergência × **transição conformacional real** |
| `AlignTraj` antes do `RMSF` | O referencial da análise: movimento interno × movimento pela caixa |

# Exercícios (entrega até a Aula 8)

1. **(Obrigatório)** Escolha uma proteína da **sua** linha de pesquisa. Verifique se há estrutura no PDB; se houver, registre código, método, resolução, R-free **e número de cadeias**. Se não houver, gere um modelo no **SWISS-MODEL** *e* busque no **AlphaFold DB**, e compare GMQE/QMEANDisCo com o pLDDT. Entregue meia página justificando qual estrutura você usaria.

2. **(Obrigatório)** Na célula C4, mude `EXHAUSTIVENESS` para **8, 16 e 32**, e para cada valor rode **três vezes** com sementes diferentes (`SEMENTE = 1, 2, 3`). Monte uma tabela com **média e desvio-padrão** da melhor afinidade. A partir de qual valor o resultado estabiliza? O que isso diz sobre reportar um único número com duas casas decimais?

3. **(Obrigatório)** Compare a ordem dos scores da triagem (C7) com os valores de **IC50** publicados — busque cada composto no **ChEMBL**. A ordem bate? Escreva um parágrafo discutindo as causas de divergência, incluindo o efeito do tamanho do ligante e a *ligand efficiency* que você calculou. Se algum ligante foi recusado pelas conferências da célula C2, pegue o SMILES na página do PubChem, preencha o campo `reserva` e inclua-o na comparação.

4. **(Obrigatório, diagnóstico)** Rode a célula **C8** e relate: quantos átomos o "ligante cristalográfico" passou a ter, para onde foi o centro da caixa, qual a afinidade reportada e qual o RMSD do redocking. **Descreva como você teria descoberto o erro se ninguém tivesse avisado.**

5. **(Desafio)** Use o script **`vina_docking.py`** (<https://github.com/robertoaabfilho/vina_docking>) e o **`convert_to_pdbqt.py`** (<https://github.com/robertoaabfilho/to_pdbqt>) para dockar um conjunto de **20 ligantes** do PubChem contra a **4EY7** e contra a **butirilcolinesterase** (PDB **1P0I**). Discuta a seletividade entre as duas enzimas — o que isso significa para um *bioscavenger*?

6. **(Desafio)** No Bloco D, calcule o número de **ligações de hidrogênio intramoleculares** ao longo do tempo (`MDAnalysis.analysis.hydrogenbonds`) e verifique se a estrutura secundária se mantém durante a abertura dos domínios. Isso **confirma ou refuta** a leitura de "abertura, não desenovelamento"?

7. **(Desafio)** Na célula C6, aumente o `RAIO_CONTATO` de 4,0 para 5,0 Å. Quantos resíduos entram? A fração de contatos coincidentes com o cristal melhora ou piora? Por quê?

---

## Referências das ferramentas e dos dados

**Dados públicos usados**

- **PDB 4EY7** — Cheung, J. *et al.* Structures of human acetylcholinesterase in complex with pharmacologically important ligands. *J. Med. Chem.*, 2012.
- **AlphaFold DB, P22303** — Jumper, J. *et al.* Highly accurate protein structure prediction with AlphaFold. *Nature*, 2021. · Varadi, M. *et al.* AlphaFold Protein Structure Database. *Nucleic Acids Res.*, 2022.
- **Trajetória da adenilato quinase** — conjunto de dados de teste do MDAnalysis (estrutura de partida PDB **4AKE**).
- **PubChem** CIDs 3152, 9651, 1935.

**Programas (todos livres ou gratuitos para uso acadêmico)**

- **AutoDock Vina** — Trott, O.; Olson, A. J. *J. Comput. Chem.*, 2010. · Eberhardt, J. *et al.* AutoDock Vina 1.2.0. *J. Chem. Inf. Model.*, 2021.
- **Open Babel** — O'Boyle, N. M. *et al.* Open Babel: An open chemical toolbox. *J. Cheminformatics*, 2011.
- **RDKit** — <https://www.rdkit.org>
- **Biopython** — Cock, P. J. A. *et al.* *Bioinformatics*, 2009.
- **MDAnalysis** — Michaud-Agrawal, N. *et al.* *J. Comput. Chem.*, 2011. · Gowers, R. J. *et al.* *Proc. SciPy*, 2016.

*Material da disciplina Introdução à Bioinformática — PPGED/IME. Dados e ferramentas de uso público e livre.*
